## Ricavo funzione
in questo codice ricavo la funzione tramite il fit dei dati.

In [ ]:
import ROOT
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati
ROOT.gStyle.SetFitFormat(
    "4.2e"
)  # con 2 cifre decimali

In [ ]:
#fit 
def Malus(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    return A*np.cos(t - C)**2 + B

def Malus_fondo(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    D = par[3]
    return (A*np.cos(t - C)**2 + B)*(1 + D*t) 

def modellopy(t, A, B, C):         
    return A*np.cos(t - C)**2 + B

def modellopy_fondo(t, A, B, C, D):         
    return (A*np.cos(t - C)**2 + B)* (1+ D*t )

def Malus_deriv(x, par):
    t = x
    dt = 1e-6
    return ((modellopy(t+dt/2, par[0], par[1], par[2])-modellopy(t-dt/2, par[0], par[1], par[2]))/dt)

def Malus_deriv_fondo(x, par):
    t = x
    dt = 1e-6
    return ((modellopy_fondo(t+dt/2, par[0], par[1], par[2], par[3])-modellopy_fondo(t-dt/2, par[0], par[1], par[2], par[3]))/dt)   

In [ ]:
xlsx ="Esperienza4.xlsx"
df = pd.read_excel(xlsx, sheet_name= "Dati")
df = df.dropna(subset=["θ [rad]", "V", "sigma_θ", "sigma_V"])
df = df.to_numpy(dtype = 'float64')

x = df[:, 1]     
y = df[:, 2]     

err_x = df[:, 3]     
err_y = df[:, 4]   

# eliminazione primi dati
index = [i for i in range(6)]

x = np.delete(x, index)
y = np.delete(y, index)
err_x = np.delete(err_x, index)
err_y = np.delete(err_y, index)

In [ ]:
#minimizzazzione con ROOT
MalusFit = ROOT.TF1('model', Malus, min(x), max(x), 3)
MalusFit.SetParameters(y.max() - y.min(), y.min(), x[np.argmax(y)])     # A è Vmax-Vmin, B è Vmin, C è posizione del massimo
MalusFit.SetParNames("A", "B", "C")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati sperimentali e fit; #theta [rad]; V [V]")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(MalusFit, "S") # SR significa che voglio fare il fit senza disegnare la funzione e senza stampare a video i risultati del fit 

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = MalusFit.GetParameters()

errors = MalusFit.GetParErrors()

chi2 = fit_res.Chi2()
ndf = fit_res.Ndf()

err_yeff = np.sqrt(err_y**2 + ((Malus_deriv(x, par)) * err_x)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"C = {par[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}, ndf = {ndf}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.14, 0.14, 0.42, 0.23)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(MalusFit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("Immagini/malus_fit.png")

## Confronto
In questa cella confronto se la funzione che ho trovato è quella che vado a togliere al mio fit e quindi ho effettivamente un polarizzazione della sorgente (guardo anche se corrispondono i valori.)

In [ ]:
# funzione di fit per correggere disomogeneità del polarizzatore
def Malus_filtro(x, par):
    t = x[0]
    A = par[0]
    B = par[1]
    C = par[2]
    D = par[3]
    E = par[4]
    F = par[5]
    return (A*np.cos(t - C)**2 + B)* (1 + D*t) *(1 + E*np.cos(2*t - F))

def modellopy_filtroF(t, A, B, C, D, E, F):         
    return (A*np.cos(t - C)**2 + B)* (1+ D*t ) *(1 + E*np.cos(2*t - F))

def Malus_deriv_filtroF(x, par):
    t = x
    dt = 1e-6
    return ((modellopy_filtroF(t+dt/2, par[0], par[1], par[2], par[3], par[4], par[5])-modellopy_filtroF(t-dt/2, par[0], par[1], par[2], par[3], par[4], par[5]))/dt)

In [ ]:
#minimizzazzione con ROOT
MalusFit_filtroF = ROOT.TF1('model', Malus_filtro, min(x), max(x), 6)
MalusFit_filtroF.SetParameters(par_fondo[0], par_fondo[1], par_fondo[2], par_fondo[3],0, par_fondo[2])
MalusFit_filtroF.SetParNames("A", "B", "C", "D", "E", "F")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati sperimentali e fit; #theta [rad]; V [V]")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(MalusFit_filtroF, "S") # SR significa che voglio fare il fit senza disegnare la funzione e senza stampare a video i risultati del fit 

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par_filtroF = MalusFit_filtroF.GetParameters()

errors_filtroF = MalusFit_filtroF.GetParErrors()

chi2F = fit_res.Chi2()
ndfF = fit_res.Ndf()

err_yeff = np.sqrt(err_y**2 + ((Malus_deriv_filtroF(x, par_filtroF)) * err_x)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par_filtroF[0]} ± {errors_filtroF[0]}")
print(f"B = {par_filtroF[1]} ± {errors_filtroF[1]}")
print(f"C = {par_filtroF[2]} ± {errors_filtroF[2]}")
print(f"D = {par_filtroF[3]} ± {errors_filtroF[3]}")
print(f"E = {par_filtroF[4]} ± {errors_filtroF[4]}")
print(f"F = {par_filtroF[5]} ± {errors_filtroF[5]}")
print(f"Chi2 = {chi2F}, ndf = {ndfF}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.14, 0.14, 0.42, 0.23)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(MalusFit_filtroF, "Fit", "l")
leg1.Draw()
c1.Draw()
c1.SaveAs("Immagini/malus_fit_filtro.png")